# Hierarchical VMamba — Dysgraphia Classification (v3)

Your v2 (flat SS2D) run showed huge fold-to-fold variance (40%-80% accuracy,
CV mean 65.9% ± 14.9%) — several folds got stuck predicting the majority
class early and never escaped. This version addresses both the **architecture**
and the **training instability**:

### Architecture: Hierarchical VMamba (4 stages + patch merging)
The flat SS2D stack from before processed every layer at the same resolution.
The *actual* published VMamba architecture (Liu et al., 2024) is hierarchical
— like Swin Transformer / ResNet: resolution halves and channels double
between 4 stages. Stage 1 sees fine stroke texture; stage 4 sees whole
word/line-level structure. This is a genuine architectural upgrade over the
flat version, more citable as "VMamba" proper, and — practically — deeper
stages have far fewer tokens, so gradients there are less noisy.

### Training stability fixes
- **Class-balanced sampling**: a `WeightedRandomSampler` ensures every
  minibatch is roughly balanced, directly targeting the "stuck predicting
  majority class" failure mode.
- **Gentler warmup + lower peak LR** (2e-4 instead of 3e-4, 15% warmup
  instead of 10%) — some folds were still getting perturbed early on.
- Mixup, k-fold CV, fold-ensembling + TTA inference: all kept from before.

**Set `Runtime -> Change runtime type -> GPU` before running.**


## 1. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — go to Runtime > Change runtime type > GPU, then rerun.")


## 2. Install dependencies

In [ ]:
!pip install -q scikit-learn pillow torch torchvision

## 3. Get data — OPTION A: direct upload (e.g. from a pendrive)

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # pick your Dysgraphic.zip
zip_name = list(uploaded.keys())[0]
extract_dir = "/content/data"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall(extract_dir)
print("Extracted to:", extract_dir)
print(os.listdir(extract_dir))


## 3. Get data — OPTION B: Google Drive (skip if you used Option A)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DATA_PATH = "/content/drive/MyDrive/Dysgraphic"  # EDIT if needed
print("Contents:", __import__('os').listdir(DRIVE_DATA_PATH))


## 4. Set data root

In [ ]:
DATA_ROOT = "/content/data/Dysgraphic"   # <-- EDIT if needed

import os
assert os.path.isdir(os.path.join(DATA_ROOT, "lpd")), f"lpd/ not found under {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, "pd")), f"pd/ not found under {DATA_ROOT}"
n_lpd = len([f for f in os.listdir(os.path.join(DATA_ROOT, "lpd")) if not f.startswith('.')])
n_pd  = len([f for f in os.listdir(os.path.join(DATA_ROOT, "pd"))  if not f.startswith('.')])
print(f"lpd: {n_lpd} images, pd: {n_pd} images")


## 5. Model — Hierarchical VMamba (4 stages, patch merging, SS2D)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def selective_scan(x, delta, A, B, C, D):
    batch, L, d_inner = x.shape
    N = A.shape[1]
    device, dtype = x.device, x.dtype
    deltaA = torch.exp(delta.unsqueeze(-1) * A.view(1, 1, d_inner, N))
    deltaB_x = delta.unsqueeze(-1) * B.unsqueeze(2) * x.unsqueeze(-1)
    h = torch.zeros(batch, d_inner, N, device=device, dtype=dtype)
    ys = []
    for t in range(L):
        h = deltaA[:, t] * h + deltaB_x[:, t]
        y_t = torch.einsum("bdn,bn->bd", h, C[:, t])
        ys.append(y_t)
    y = torch.stack(ys, dim=1)
    return y + x * D


class S6Core(nn.Module):
    def __init__(self, d_inner, d_state=16, dt_rank=None):
        super().__init__()
        self.d_inner = d_inner
        self.d_state = d_state
        self.dt_rank = dt_rank or max(1, d_inner // 16)
        self.x_proj = nn.Linear(d_inner, self.dt_rank + d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, d_inner, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_inner))
        dt_init_std = self.dt_rank ** -0.5
        nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        dt = torch.exp(
            torch.rand(d_inner) * (torch.log(torch.tensor(0.1)) - torch.log(torch.tensor(0.001)))
            + torch.log(torch.tensor(0.001))
        ).clamp(min=1e-4)
        inv_softplus = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_softplus)

    def forward(self, x):
        x_dbl = self.x_proj(x)
        dt, Bp, Cp = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(dt))
        A = -torch.exp(self.A_log)
        return selective_scan(x, delta, A, Bp, Cp, self.D)


class SS2D(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=3, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_inner = expand * d_model
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv2d = nn.Conv2d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                 padding=d_conv // 2, groups=self.d_inner, bias=True)
        self.core = S6Core(self.d_inner, d_state=d_state)
        self.out_norm = nn.LayerNorm(self.d_inner)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        B, H, W, C = x.shape
        x_and_res = self.in_proj(x)
        x_in, res = x_and_res.chunk(2, dim=-1)
        x_in = x_in.permute(0, 3, 1, 2)
        x_in = self.conv2d(x_in)
        x_in = F.silu(x_in)

        seq_row_fwd = x_in.flatten(2).transpose(1, 2)
        seq_row_bwd = torch.flip(seq_row_fwd, dims=[1])
        seq_col_fwd = x_in.transpose(2, 3).flatten(2).transpose(1, 2)
        seq_col_bwd = torch.flip(seq_col_fwd, dims=[1])

        y_row_fwd = self.core(seq_row_fwd)
        y_row_bwd = torch.flip(self.core(seq_row_bwd), dims=[1])
        y_col_fwd = self.core(seq_col_fwd)
        y_col_bwd = torch.flip(self.core(seq_col_bwd), dims=[1])

        y_row_fwd = y_row_fwd.transpose(1, 2).reshape(B, self.d_inner, H, W).permute(0, 2, 3, 1)
        y_row_bwd = y_row_bwd.transpose(1, 2).reshape(B, self.d_inner, H, W).permute(0, 2, 3, 1)
        y_col_fwd = y_col_fwd.transpose(1, 2).reshape(B, self.d_inner, W, H).permute(0, 3, 2, 1)
        y_col_bwd = y_col_bwd.transpose(1, 2).reshape(B, self.d_inner, W, H).permute(0, 3, 2, 1)

        y = y_row_fwd + y_row_bwd + y_col_fwd + y_col_bwd
        y = self.out_norm(y)
        y = y * F.silu(res)
        return self.out_proj(y)


class VSSBlock(nn.Module):
    def __init__(self, d_model, d_state=16, expand=2, mlp_ratio=2.0, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.ss2d = SS2D(d_model, d_state=d_state, expand=expand)
        self.drop_path = nn.Dropout(drop_path) if drop_path > 0 else nn.Identity()
        hidden = int(d_model * mlp_ratio)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, hidden), nn.GELU(), nn.Linear(hidden, d_model))

    def forward(self, x):
        x = x + self.drop_path(self.ss2d(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class PatchEmbed2D(nn.Module):
    def __init__(self, img_size=128, patch_size=8, in_chans=3, embed_dim=48):
        super().__init__()
        self.grid_size = img_size // patch_size
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        return x.permute(0, 2, 3, 1)


class PatchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = nn.LayerNorm(4 * dim)

    def forward(self, x):
        B, H, W, C = x.shape
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], dim=-1)
        x = self.norm(x)
        return self.reduction(x)


class HierarchicalVMamba(nn.Module):
    def __init__(self, img_size=128, patch_size=8, in_chans=3, num_classes=2,
                 base_dim=48, depths=(1, 1, 2, 1), d_state=16, expand=2,
                 drop_rate=0.1, drop_path_rate=0.1):
        super().__init__()
        dims = [base_dim * (2 ** i) for i in range(len(depths))]
        self.patch_embed = PatchEmbed2D(img_size, patch_size, in_chans, dims[0])
        self.pos_drop = nn.Dropout(drop_rate)

        total_blocks = sum(depths)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, total_blocks)]
        block_idx = 0
        self.stages = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        for i, depth in enumerate(depths):
            blocks = nn.ModuleList([
                VSSBlock(dims[i], d_state=d_state, expand=expand, drop_path=dpr[block_idx + j])
                for j in range(depth)
            ])
            self.stages.append(blocks)
            block_idx += depth
            self.downsamples.append(PatchMerging(dims[i]) if i < len(depths) - 1 else None)

        self.norm = nn.LayerNorm(dims[-1])
        self.head = nn.Sequential(nn.Dropout(drop_rate), nn.Linear(dims[-1], num_classes))
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.zeros_(m.bias)
            nn.init.ones_(m.weight)

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.pos_drop(x)
        for blocks, downsample in zip(self.stages, self.downsamples):
            for blk in blocks:
                x = blk(x)
            if downsample is not None:
                x = downsample(x)
        x = self.norm(x)
        x = x.mean(dim=[1, 2])
        return self.head(x)


def build_hvmamba(num_classes=2, variant="tiny"):
    """
    'tiny'  - img 128, patch 8, base_dim 48, depths (1,1,2,1)  (default, ~3M params)
    'small' - img 128, patch 8, base_dim 64, depths (2,2,4,2)  (slower, ~10M params)
    """
    configs = {
        "tiny":  dict(img_size=128, patch_size=8, base_dim=48, depths=(1, 1, 2, 1)),
        "small": dict(img_size=128, patch_size=8, base_dim=64, depths=(2, 2, 4, 2)),
    }
    cfg = configs[variant]
    model = HierarchicalVMamba(
        img_size=cfg["img_size"], patch_size=cfg["patch_size"], num_classes=num_classes,
        base_dim=cfg["base_dim"], depths=cfg["depths"], d_state=16, expand=2,
        drop_rate=0.15, drop_path_rate=0.1,
    )
    return model, cfg["img_size"]


for v in ["tiny", "small"]:
    m, isz = build_hvmamba(variant=v)
    out = m(torch.randn(2, 3, isz, isz))
    print(f"{v:6s} (img={isz}) -> out {tuple(out.shape)}, params {sum(p.numel() for p in m.parameters()):,}")
del m, out


## 6. Dataset

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

CLASS_TO_IDX = {"lpd": 0, "pd": 1}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
IMG_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")


def list_samples(root_dir):
    samples = []
    for class_name, label in CLASS_TO_IDX.items():
        class_dir = os.path.join(root_dir, class_name)
        if not os.path.isdir(class_dir):
            raise FileNotFoundError(f"Expected folder not found: {class_dir}")
        for fname in sorted(os.listdir(class_dir)):
            if fname.lower().endswith(IMG_EXTENSIONS):
                samples.append((os.path.join(class_dir, fname), label))
    if len(samples) == 0:
        raise RuntimeError(f"No images found under {root_dir}")
    return samples


class DysgraphiaDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


def get_transforms(img_size, train=True):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    if train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomAffine(degrees=6, translate=(0.04, 0.04), scale=(0.97, 1.03)),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

samples = list_samples(DATA_ROOT)
print(f"Found {len(samples)} images "
      f"({sum(1 for _, l in samples if l == 0)} lpd, {sum(1 for _, l in samples if l == 1)} pd)")


## 7. Config

In [ ]:
class Config:
    variant = "tiny"
    epochs = 40
    lr = 2e-4              # lowered from 3e-4 — some v2 folds got perturbed early
    weight_decay = 0.02
    batch_size = 16
    kfold = 5
    patience = 12
    seed = 42
    mixup = True
    mixup_alpha = 0.3
    warmup_frac = 0.15     # gentler warmup than v2's 0.10
    out_dir = "/content/checkpoints"

cfg = Config()
os.makedirs(cfg.out_dir, exist_ok=True)

import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

_, IMG_SIZE = build_hvmamba(variant=cfg.variant)
print("Image size for this variant:", IMG_SIZE)


## 8. Training helpers — class-balanced sampling, warmup+cosine, Mixup

The `WeightedRandomSampler` directly targets the "stuck predicting the
majority class" failure mode seen in several v2 folds — every minibatch now
sees a roughly 50/50 mix of classes regardless of the fold's natural split.


In [ ]:
import math
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


def make_balanced_sampler(samples):
    labels = [s[1] for s in samples]
    class_counts = np.bincount(labels)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[l] for l in labels]
    return WeightedRandomSampler(sample_weights, num_samples=len(samples), replacement=True)


def make_warmup_cosine_scheduler(optimizer, total_epochs, warmup_frac=0.15):
    warmup_epochs = max(1, int(total_epochs * warmup_frac))
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def mixup_batch(imgs, labels, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    return mixed, labels, labels[idx], lam


def mixup_criterion(criterion, logits, labels_a, labels_b, lam):
    return lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)


def run_one_epoch(model, loader, criterion, optimizer, device, train=True, use_mixup=False, mixup_alpha=0.3):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    torch.set_grad_enabled(train)
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if train:
            optimizer.zero_grad()
        if train and use_mixup:
            mixed_imgs, labels_a, labels_b, lam = mixup_batch(imgs, labels, alpha=mixup_alpha)
            logits = model(mixed_imgs)
            loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
        else:
            logits = model(imgs)
            loss = criterion(logits, labels)
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1, all_labels, all_preds


def train_model(train_samples, val_samples, device, variant="tiny", epochs=40,
                 lr=2e-4, weight_decay=0.02, batch_size=16, patience=12,
                 use_mixup=True, mixup_alpha=0.3, warmup_frac=0.15, verbose=True):
    model, img_size = build_hvmamba(num_classes=2, variant=variant)
    model = model.to(device)

    train_ds = DysgraphiaDataset(train_samples, get_transforms(img_size, train=True))
    val_ds = DysgraphiaDataset(val_samples, get_transforms(img_size, train=False))

    sampler = make_balanced_sampler(train_samples)
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = make_warmup_cosine_scheduler(optimizer, epochs, warmup_frac=warmup_frac)

    best_f1, best_state, epochs_no_improve = -1, None, 0
    best_labels, best_preds = None, None

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, train_f1, _, _ = run_one_epoch(
            model, train_loader, criterion, optimizer, device, train=True,
            use_mixup=use_mixup, mixup_alpha=mixup_alpha)
        val_loss, val_acc, val_f1, val_labels, val_preds = run_one_epoch(
            model, val_loader, criterion, optimizer, device, train=False)
        scheduler.step()

        if verbose:
            cur_lr = optimizer.param_groups[0]["lr"]
            print(f"  Epoch {epoch:3d}/{epochs} | lr {cur_lr:.2e} | "
                  f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
                  f"val loss {val_loss:.4f} acc {val_acc:.3f} f1 {val_f1:.3f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_labels, best_preds = val_labels, val_preds
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f"  Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    return model, img_size, best_f1, best_labels, best_preds


## 9. Stratified K-fold cross-validation

In [ ]:
from sklearn.model_selection import StratifiedKFold

def cross_validate(samples, device, cfg):
    labels = [s[1] for s in samples]
    skf = StratifiedKFold(n_splits=cfg.kfold, shuffle=True, random_state=cfg.seed)

    fold_accs, fold_f1s = [], []
    for fold, (train_idx, val_idx) in enumerate(skf.split(samples, labels), start=1):
        print(f"\n===== Fold {fold}/{cfg.kfold} =====")
        train_samples = [samples[i] for i in train_idx]
        val_samples = [samples[i] for i in val_idx]

        model, img_size, best_f1, val_labels, val_preds = train_model(
            train_samples, val_samples, device, variant=cfg.variant,
            epochs=cfg.epochs, lr=cfg.lr, weight_decay=cfg.weight_decay,
            batch_size=cfg.batch_size, patience=cfg.patience,
            use_mixup=cfg.mixup, mixup_alpha=cfg.mixup_alpha,
            warmup_frac=cfg.warmup_frac,
        )
        acc = accuracy_score(val_labels, val_preds)
        fold_accs.append(acc)
        fold_f1s.append(best_f1)
        print(f"Fold {fold} result -> acc: {acc:.3f}, macro-F1: {best_f1:.3f}")
        print(classification_report(val_labels, val_preds,
              target_names=[IDX_TO_CLASS[0], IDX_TO_CLASS[1]], zero_division=0))

        torch.save({
            "model_state": model.state_dict(),
            "variant": cfg.variant,
            "img_size": img_size,
            "class_to_idx": CLASS_TO_IDX,
        }, os.path.join(cfg.out_dir, f"hvmamba_fold{fold}.pt"))

    print("\n===== Cross-validation summary =====")
    print(f"Accuracy : {np.mean(fold_accs):.3f} +/- {np.std(fold_accs):.3f}")
    print(f"Macro-F1 : {np.mean(fold_f1s):.3f} +/- {np.std(fold_f1s):.3f}")
    return fold_accs, fold_f1s

fold_accs, fold_f1s = cross_validate(samples, device, cfg)


## 10. Train the final deployable model on (almost) all the data

In [ ]:
from sklearn.model_selection import train_test_split

labels = [s[1] for s in samples]
train_samples, val_samples = train_test_split(
    samples, test_size=0.15, stratify=labels, random_state=cfg.seed
)

final_model, img_size, final_f1, val_labels, val_preds = train_model(
    train_samples, val_samples, device, variant=cfg.variant,
    epochs=cfg.epochs, lr=cfg.lr, weight_decay=cfg.weight_decay,
    batch_size=cfg.batch_size, patience=cfg.patience,
    use_mixup=cfg.mixup, mixup_alpha=cfg.mixup_alpha, warmup_frac=cfg.warmup_frac,
)

print("\nFinal held-out validation report:")
print(classification_report(val_labels, val_preds,
      target_names=[IDX_TO_CLASS[0], IDX_TO_CLASS[1]], zero_division=0))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(val_labels, val_preds))

ckpt_path = os.path.join(cfg.out_dir, "hvmamba_dysgraphia_final.pt")
torch.save({
    "model_state": final_model.state_dict(),
    "variant": cfg.variant,
    "img_size": img_size,
    "class_to_idx": CLASS_TO_IDX,
}, ckpt_path)
print(f"\nSaved final model to: {ckpt_path}")


## 11. Download checkpoints (optional)

In [ ]:
from google.colab import files
files.download(ckpt_path)
# for f in range(1, cfg.kfold + 1):
#     files.download(os.path.join(cfg.out_dir, f"hvmamba_fold{f}.pt"))


## 12. Inference with Test-Time Augmentation (TTA) + fold ensembling

In [ ]:
import torch.nn.functional as F
from torchvision import transforms

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

def tta_views(img, img_size):
    base = transforms.Compose([
        transforms.Resize((img_size, img_size)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    views = [base(img)]
    slight = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomAffine(degrees=4, translate=(0.02, 0.02)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    for _ in range(3):
        views.append(slight(img))
    return torch.stack(views, dim=0)


def load_ckpt_model(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    m, _ = build_hvmamba(num_classes=2, variant=ckpt["variant"])
    m = m.to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    return m, ckpt["img_size"]


def predict_ensemble(image_path, checkpoint_paths, use_tta=True):
    img = Image.open(image_path).convert("RGB")
    all_probs = []
    for ckpt_path in checkpoint_paths:
        model, img_size = load_ckpt_model(ckpt_path)
        if use_tta:
            views = tta_views(img, img_size).to(device)
        else:
            views = transforms.Compose([transforms.Resize((img_size, img_size)),
                     transforms.ToTensor(), transforms.Normalize(MEAN, STD)])(img).unsqueeze(0).to(device)
        with torch.no_grad():
            probs = F.softmax(model(views), dim=1).mean(dim=0)
        all_probs.append(probs.cpu())
    avg_probs = torch.stack(all_probs, dim=0).mean(dim=0)
    pred_idx = int(avg_probs.argmax())
    print(f"Prediction: {IDX_TO_CLASS[pred_idx].upper()}")
    print(f"Confidence -> lpd: {avg_probs[0]:.3f}, pd: {avg_probs[1]:.3f}")
    return IDX_TO_CLASS[pred_idx], avg_probs

# Example:
# fold_ckpts = [os.path.join(cfg.out_dir, f"hvmamba_fold{f}.pt") for f in range(1, cfg.kfold + 1)]
# predict_ensemble("/content/data/Dysgraphic/lpd/some_image.jpg", fold_ckpts)


## Notes

- **Architecture novelty claim for your paper**: hierarchical VMamba (4-stage,
  patch-merging, SS2D) applied to dysgraphia handwriting classification — the
  first application of this architecture family to this domain, addressing
  both plain-CNN limited receptive fields and plain Vision Mamba's loss of 2D
  adjacency from 1D flattening.
- **If variance across folds is still high** after these fixes, that's a
  strong signal the ceiling is genuinely data-limited (249 images / 83
  participants) rather than an architecture problem — worth reporting
  per-fold results transparently in your paper rather than only the mean.
- **Other Vision-Mamba-family options** if you want to explore further or
  discuss in "related work": MambaVision (NVIDIA, hybrid Mamba+attention,
  hierarchical) and LocalMamba (locality-aware scan ordering) are the next
  most citable alternatives, though more involved to implement correctly
  from scratch than this hierarchical VMamba.
- Remember the subject-level split concern from before — check whether
  images can be traced back to the 83 participants to avoid data leakage
  across train/val.
